In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, Counter
from sklearn.cluster import KMeans
import os

def crop_torso(frame, box):
    """Crops the middle 50% of the bounding box to isolate the jersey."""
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    torso_y1 = y1 + int(0.25 * h)
    torso_y2 = y1 + int(0.75 * h)
    return frame[torso_y1:torso_y2, x1:x2]

def jersey_feature(img):
    """Extracts a 3D BGR histogram, masking out the beige floor and shadows."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    
    lower_beige = np.array([10, 20, 100])
    upper_beige = np.array([40, 150, 255])
    beige_mask = cv2.inRange(hsv, lower_beige, upper_beige)

    lower_dark = np.array([0, 0, 0])
    upper_dark = np.array([180, 255, 50])
    dark_mask = cv2.inRange(hsv, lower_dark, upper_dark)

    ignore_mask = cv2.bitwise_or(beige_mask, dark_mask)
    valid_mask = cv2.bitwise_not(ignore_mask)

    if cv2.countNonZero(valid_mask) < 50:
        return None

  
    hist = cv2.calcHist(
        [img], [0, 1, 2], valid_mask, 
        [8, 8, 8], [0, 256, 0, 256, 0, 256]
    )
    cv2.normalize(hist, hist)
    return hist.flatten()

model = YOLO("yolov8n.pt")
video_path = "video_4.mp4"

player_features = defaultdict(list)
tracking_history = [] 

print("Pass 1: Tracking players and extracting jersey colors...")
results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    persist=True,
    stream=True 
)

for r in results:
    frame = r.orig_img 
    frame_info = {'boxes': [], 'track_ids': []}
    
    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy()
        class_ids = r.boxes.cls.cpu().numpy()

        for box, track_id, cls in zip(boxes, track_ids, class_ids):
            if int(cls) != 0: # Assuming 0 is 'person'/'player'
                continue
            
            frame_info['boxes'].append(box)
            frame_info['track_ids'].append(track_id)
            
            torso_crop = crop_torso(frame, box)
            if torso_crop.size > 0:
                feat = jersey_feature(torso_crop)
                if feat is not None:
                    player_features[int(track_id)].append(feat)
                
    tracking_history.append(frame_info)

print("Clustering players into teams...")
player_embeddings = {
    tid: np.mean(feats, axis=0)
    for tid, feats in player_features.items()
    if len(feats) > 10  
}

X = np.array(list(player_embeddings.values()))
track_ids = list(player_embeddings.keys())

kmeans = KMeans(n_clusters=2, random_state=0, n_init="auto").fit(X)
track_to_team = {tid: label for tid, label in zip(track_ids, kmeans.labels_)}

counts = Counter(track_to_team.values())
TEAM_A, TEAM_B = list(counts.keys())[:2]

print("Pass 2: Writing output video...")
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_path = "data/output_videos/team_assignment.mp4"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
assert out.isOpened(), "Failed to open VideoWriter"

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    if frame_idx < len(tracking_history):
        data = tracking_history[frame_idx]
        
        for box, track_id in zip(data['boxes'], data['track_ids']):
            tid = int(track_id)
            
            if tid not in track_to_team:
                continue
                
            team = track_to_team[tid]
            
            color = (255, 0, 0) if team == TEAM_A else (255, 255, 255)
            text_color = (255, 255, 255) if team == TEAM_A else (0, 0, 0)
            
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            label = f"Team {'A' if team == TEAM_A else 'B'}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - 20), (x1 + tw, y1), color, -1)
            cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)
    
    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"Done! Video saved to {output_path}")

Pass 1: Tracking players and extracting jersey colors...

video 1/1 (frame 1/357) /Users/alysayao/Sports DAML project/Computer-Vision-Project/video_4.mp4: 384x640 7 persons, 48.9ms
video 1/1 (frame 2/357) /Users/alysayao/Sports DAML project/Computer-Vision-Project/video_4.mp4: 384x640 6 persons, 46.8ms
video 1/1 (frame 3/357) /Users/alysayao/Sports DAML project/Computer-Vision-Project/video_4.mp4: 384x640 5 persons, 47.2ms
video 1/1 (frame 4/357) /Users/alysayao/Sports DAML project/Computer-Vision-Project/video_4.mp4: 384x640 6 persons, 54.8ms
video 1/1 (frame 5/357) /Users/alysayao/Sports DAML project/Computer-Vision-Project/video_4.mp4: 384x640 5 persons, 50.1ms
video 1/1 (frame 6/357) /Users/alysayao/Sports DAML project/Computer-Vision-Project/video_4.mp4: 384x640 6 persons, 48.6ms
video 1/1 (frame 7/357) /Users/alysayao/Sports DAML project/Computer-Vision-Project/video_4.mp4: 384x640 6 persons, 48.7ms
video 1/1 (frame 8/357) /Users/alysayao/Sports DAML project/Computer-Vision-Proje